# 01 · Ghost Games: Headline Results

**Question:** When COVID emptied stadiums, how much of football's home advantage disappeared?

Steps: load results → add real attendance → label each match (fans / partial / empty) → raw comparison → model with team strength → home advantage over time.

In [ ]:
import matplotlib.pyplot as plt

from ghost_games.attendance import (
    add_crowd_share,
    apply_overrides,
    load_attendance,
    match_team_names,
    merge_attendance,
)
from ghost_games.data import load_matches
from ghost_games.inference import bootstrap_gap, permutation_test
from ghost_games.model import fit_home_advantage, home_advantage_over_time
from ghost_games.outcomes import add_outcomes
from ghost_games.treatment import (
    assign_crowd_status_from_attendance,
    compare_with_dates,
    load_windows,
)

results = load_matches("../data/raw")
attendance = load_attendance("../data/raw/attendance")
len(results), len(attendance)

## 1. Match team names between the two sources
Check every row with `date_overlap` below 0.9. Put any fixes in `config/team_names.csv`.

In [ ]:
mapping = apply_overrides(match_team_names(results, attendance), "../config/team_names.csv")
mapping[mapping["date_overlap"].fillna(1) < 0.9]

In [ ]:
matches = add_crowd_share(merge_attendance(results, attendance, mapping))
print(f"Attendance found for {matches.attrs['attendance_coverage']:.1%} of matches")
windows = load_windows("../config/closed_doors.csv")
matches = add_outcomes(assign_crowd_status_from_attendance(matches, windows))
matches.groupby(["season", "crowd_status"]).size().unstack(fill_value=0)

## 2. Do the config dates agree with real attendance?
Numbers off the diagonal mean a date window is wrong.

In [ ]:
compare_with_dates(matches)

## 3. Raw comparison (no controls)

In [ ]:
main = matches[matches.crowd_status != "partial"]
for outcome in ["goal_diff", "points_diff", "card_gap"]:
    fans = main.loc[main.closed_doors == 0, outcome]
    empty = main.loc[main.closed_doors == 1, outcome]
    gap = bootstrap_gap(fans, empty)
    p = permutation_test(fans, empty, n_permutations=2000)
    print(
        f"{outcome:12s} fans={gap.fans_mean:+.3f} empty={gap.empty_mean:+.3f} "
        f"diff={gap.difference:+.3f} [{gap.ci_low:+.3f}, {gap.ci_high:+.3f}] p={p:.4f}"
    )

## 4. Controlling for team strength
First fans vs empty, then crowd size as a continuous variable (uses the partial-crowd games too).

In [ ]:
for treatment in ["closed_doors", "crowd_share"]:
    print(f"--- {treatment}")
    for outcome in ["goal_diff", "points_diff", "card_gap"]:
        fit = fit_home_advantage(matches, outcome, treatment=treatment)
        print(
            f"{outcome:12s} home_adv={fit.home_adv:+.3f} effect={fit.effect:+.3f} "
            f"CI={fit.ci[0]:+.3f}..{fit.ci[1]:+.3f} n={fit.n_matches} "
            f"share of edge from crowd={fit.share_of_advantage_lost:.0%}"
        )

## 5. Home advantage over time

In [ ]:
summary = home_advantage_over_time(matches, "goal_diff", freq="QS")
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(summary["date"], summary["mean"], marker="o")
ax.fill_between(summary["date"], summary["ci_low"], summary["ci_high"], alpha=0.2)
ax.axhline(0, color="grey", linewidth=0.8)
empty_quarters = summary[summary["share_empty"] > 0.5]["date"]
if not empty_quarters.empty:
    ax.axvspan(
        empty_quarters.min(),
        empty_quarters.max(),
        color="red",
        alpha=0.08,
        label="Mostly empty stadiums",
    )
ax.set_title("Home goal advantage by quarter")
ax.set_xlabel("Quarter")
ax.set_ylabel("Mean home goal difference")
ax.legend()
plt.show()
# Takeaway: TODO